# Traffic Sign YOLO Training - v1

GPU-optimized training with minimal validation for maximum speed

**Features:**
- Single-stage training with YOLO12n model
- Divergence monitoring (information only)
- Checkpoint saving every epoch
- Resume from last epoch if interrupted
- GPU optimized for A100/CUDA (mixed precision training)
- Validation enabled for monitoring training progress
- Full visualizations enabled for academic work (plots, JSON, confidences, cropped detections)

**Execution Order:**
Run sections sequentially from 1 to 10. All sections are numbered in logical order:
1. Install Dependencies
2. Imports & Setup
3. Helper Functions
4. Upload Yolo Dataset
5. Verify Dataset Structure & Count Files
6. Upload Model File
7. Configuration
8. Divergence Monitor Class
9. Training Function
10. Run Training
11. Validation with Best Model
12. Backup Results to Google Drive


In [9]:
import os
import shutil

print("🗑️  CLEANUP: Removing all existing dataset files...\n")

# Paths to remove
paths_to_remove = [
    "/content/yolo_dataset",
    "/content/yolo_dataset.zip"
]

for path in paths_to_remove:
    if os.path.exists(path):
        if os.path.isfile(path):
            os.remove(path)
            print(f"✅ Deleted file: {path}")
        elif os.path.isdir(path):
            shutil.rmtree(path)
            print(f"✅ Deleted folder: {path}")
    else:
        print(f"⏭️  Not found (skipping): {path}")

print("\n✅ Cleanup complete! Ready for fresh extraction.\n")

🗑️  CLEANUP: Removing all existing dataset files...

✅ Deleted folder: /content/yolo_dataset
✅ Deleted file: /content/yolo_dataset.zip

✅ Cleanup complete! Ready for fresh extraction.



## Section 1: Install Dependencies

**Purpose:**
* Install required Python packages for YOLO training

**Details:**
* Installs ultralytics (YOLO framework), pandas (data handling), numpy (numerical operations), torch (PyTorch), and torchvision
* Uses `--quiet` flag to suppress verbose output during installation
* Run this cell once at the beginning of your Colab session


In [1]:
# Install required packages (run once)
%pip install ultralytics pandas numpy torch torchvision --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 73.4 MB/s eta 0:00:00


## Section 2: Imports & Setup

**Purpose:**
* Import all necessary libraries and verify GPU availability

**Details:**
* Imports core libraries: os, torch, pandas, numpy, Path, YOLO
* Displays PyTorch version and CUDA availability (prioritizes A100 GPU)
* Shows GPU name and memory if CUDA is available
* Verifies that the environment is ready for A100 GPU training


In [2]:
# Import libraries
import os
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from ultralytics import YOLO

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_memory:.2f} GB")
    if "A100" in gpu_name:
        print(f"🚀 A100 GPU detected - Optimal for training!")
    else:
        print(f"💡 For best performance, consider using A100 GPU")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print(f"MPS (Apple Silicon GPU) available")
    print(f"💡 For best performance, consider using A100 GPU in Colab")
else:
    print("⚠️  No GPU available, will use CPU (training will be slow)")
    print("💡 For best performance, enable GPU runtime (preferably A100)")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch version: 2.8.0+cu126
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
GPU Memory: 42.47 GB
🚀 A100 GPU detected - Optimal for training!


## Section 3: Helper Functions

**Purpose:**
* Define utility functions used throughout the training pipeline

**Details:**
* `verify_dataset_structure(dataset_path)`: Verifies YOLO dataset structure (dataset.yaml, images/labels directories)
* `clear_gpu_cache()`: Clears GPU cache (CUDA) and system memory to prevent memory leaks
* `delete_cache_files()`: Removes YOLO dataset cache files to ensure fresh data loading
* **IMPORTANT:** Run this section FIRST before Section 4 (Upload Dataset) which uses these functions


In [3]:
def verify_dataset_structure(dataset_path):
    """Verify that the dataset has the correct YOLO structure"""
    required_dirs = [
        "images/train",
        "images/val",
        "labels/train",
        "labels/val"
    ]

    dataset_yaml = os.path.join(dataset_path, "dataset.yaml")

    print(f"\n🔍 Verifying dataset structure at: {dataset_path}")
    print("="*60)

    # Check dataset.yaml
    if os.path.exists(dataset_yaml):
        print(f"✅ Found dataset.yaml")
        # Read and display basic info
        try:
            with open(dataset_yaml, 'r') as f:
                content = f.read()
                if 'nc:' in content:
                    print(f"   Dataset YAML appears valid")
        except:
            pass
    else:
        print(f"❌ Missing dataset.yaml")
        return False

    # Check required directories
    all_exist = True
    for dir_path in required_dirs:
        full_path = os.path.join(dataset_path, dir_path)
        if os.path.exists(full_path):
            # Count files
            file_count = len([f for f in os.listdir(full_path) if os.path.isfile(os.path.join(full_path, f))])
            print(f"✅ {dir_path}: {file_count} files")
        else:
            print(f"❌ Missing: {dir_path}")
            all_exist = False

    print("="*60)

    if all_exist:
        print("✅ Dataset structure is valid!")
        return True
    else:
        print("❌ Dataset structure is incomplete!")
        return False

def clear_gpu_cache():
    """Clear CUDA cache and system memory aggressively"""
    # Clear CUDA cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        print("✅ CUDA cache cleared")

    # Force garbage collection
    import gc
    gc.collect()
    print("✅ System memory cleaned")

def delete_cache_files():
    """Delete YOLO cache files"""
    dataset_path = Path(DATASET_BASE)
    cache_files = list(dataset_path.rglob("*.cache"))

    for cache_file in cache_files:
        try:
            cache_file.unlink()
            print(f"🗑️  Deleted: {cache_file.name}")
        except Exception as e:
            print(f"❌ Failed to delete {cache_file.name}: {e}")

    if cache_files:
        print(f"✅ Cleared {len(cache_files)} cache file(s)")


## Section 4 — Upload Yolo Dataset

### **Purpose**

Efficiently load the YOLO dataset into Colab by copying a ZIP file from Google Drive and extracting it locally for fast training.

### **What this section does**

* **Section 4.1 — Copy Dataset ZIP from Drive**

  * Mounts Google Drive
  * Copies `yolo_dataset.zip` → `/content/` with progress
  * Much faster & safer than uploading large files manually
  * Verifies the ZIP exists in Drive before copying

* **Section 4.2 — Extract Dataset ZIP**

  * Extracts ZIP into `/content/yolo_dataset`
  * Shows clean percentage progress: `123/60000 (0.20%)`
  * Removes old extracted folders to avoid conflicts
  * Verifies file counts after extraction

* **Section 4.3 — Validate Dataset Structure**

  * Confirms `dataset.yaml` exists
  * Confirms required YOLO folder structure is present
  * Ensures the dataset is ready for training

### **Required ZIP Structure**

Your ZIP must contain:

```
dataset.yaml
images/
    train/
    val/
    test/       (optional)
labels/
    train/
    val/
    test/       (optional)
```

In [10]:
import os
import shutil
import zipfile
from google.colab import drive

# ----------------------------------------------------
# CONFIG
# ----------------------------------------------------
MOUNT_PATH = "/content/drive"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/yolo_dataset/yolo_dataset.zip"
LOCAL_ZIP_PATH = "/content/yolo_dataset.zip"
EXTRACT_PATH = "/content/yolo_dataset"
# ----------------------------------------------------


# ----------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ----------------------------------------------------
def mount_drive():
    print("🔗 Mounting Google Drive...")
    drive.mount(MOUNT_PATH, force_remount=True)
    print("✅ Google Drive mounted!\n")


# ----------------------------------------------------
# 2. COPY ZIP FROM DRIVE → LOCAL SESSION (WITH PROGRESS)
# ----------------------------------------------------
def copy_zip_with_progress(src, dst):
    """Copy a large ZIP file from Drive to RAM disk with progress updates."""
    if not os.path.exists(src):
        print(f"❌ ERROR: ZIP file NOT FOUND at:\n   {src}")
        return False

    file_size = os.path.getsize(src)
    copied = 0

    print(f"📦 ZIP size: {file_size/1024/1024:.2f} MB\n")
    print("🚀 Copying ZIP to local session...\n")

    with open(src, 'rb') as fsrc, open(dst, 'wb') as fdst:
        while True:
            chunk = fsrc.read(1024 * 1024)  # 1 MB chunks
            if not chunk:
                break

            fdst.write(chunk)
            copied += len(chunk)

            percent = (copied / file_size) * 100
            print(f"➡️  Copied {copied}/{file_size} bytes ({percent:.2f}%)", end="\r")

    print("\n\n✅ ZIP copy completed!\n")
    return True


# ----------------------------------------------------
# 3. EXTRACT ZIP WITH PERCENTAGE PROGRESS
# ----------------------------------------------------
def extract_zip_with_progress(zip_path, extract_to):
    """Extract ZIP file with percentage progress output."""
    if not os.path.exists(zip_path):
        print(f"❌ ZIP not found: {zip_path}")
        return False

    print(f"📂 Extracting ZIP to: {extract_to}\n")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        file_list = zip_ref.infolist()
        total = len(file_list)

        print(f"📊 Total files inside ZIP: {total}\n")

        for i, file in enumerate(file_list, start=1):
            zip_ref.extract(file, extract_to)

            # Print progress every 200 files or at the end
            if i % 200 == 0 or i == total:
                percent = (i / total) * 100
                print(f"➡️ Extracted {i}/{total} ({percent:.2f}%)")

    print("\n✅ Extraction complete!\n")
    return True


# ----------------------------------------------------
# 4. VERIFY EXTRACTION
# ----------------------------------------------------
def count_files(path):
    total = 0
    for root, dirs, files in os.walk(path):
        total += len(files)
    return total


def verify_extraction():
    extracted_count = count_files(EXTRACT_PATH)
    print(f"📌 Total extracted files: {extracted_count}\n")


# ----------------------------------------------------
# MAIN EXECUTION
# ----------------------------------------------------
def load_yolo_dataset():
    mount_drive()

    # Step 1 — Copy ZIP to local session
    success = copy_zip_with_progress(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)
    if not success:
        return

    # Step 2 — Remove old extracted folder if exists
    if os.path.exists(EXTRACT_PATH):
        print("♻️ Removing old extracted dataset...")
        shutil.rmtree(EXTRACT_PATH)

    # Step 3 — Extract ZIP with progress
    extract_zip_with_progress(LOCAL_ZIP_PATH, EXTRACT_PATH)

    # Step 4 — Verification
    verify_extraction()
    print("🎉 YOLO dataset is ready to use!\n")


# ----------------------------------------------------
# RUN
# ----------------------------------------------------
load_yolo_dataset()


🔗 Mounting Google Drive...
Mounted at /content/drive
✅ Google Drive mounted!

📦 ZIP size: 2571.96 MB

🚀 Copying ZIP to local session...

➡️  Copied 2696900094/2696900094 bytes (100.00%)

✅ ZIP copy completed!

📂 Extracting ZIP to: /content/yolo_dataset

📊 Total files inside ZIP: 120030

➡️ Extracted 200/120030 (0.17%)
➡️ Extracted 400/120030 (0.33%)
➡️ Extracted 600/120030 (0.50%)
➡️ Extracted 800/120030 (0.67%)
➡️ Extracted 1000/120030 (0.83%)
➡️ Extracted 1200/120030 (1.00%)
➡️ Extracted 1400/120030 (1.17%)
➡️ Extracted 1600/120030 (1.33%)
➡️ Extracted 1800/120030 (1.50%)
➡️ Extracted 2000/120030 (1.67%)
➡️ Extracted 2200/120030 (1.83%)
➡️ Extracted 2400/120030 (2.00%)
➡️ Extracted 2600/120030 (2.17%)
➡️ Extracted 2800/120030 (2.33%)
➡️ Extracted 3000/120030 (2.50%)
➡️ Extracted 3200/120030 (2.67%)
➡️ Extracted 3400/120030 (2.83%)
➡️ Extracted 3600/120030 (3.00%)
➡️ Extracted 3800/120030 (3.17%)
➡️ Extracted 4000/120030 (3.33%)
➡️ Extracted 4200/120030 (3.50%)
➡️ Extracted 4400/12003

## Section 5: Verify Dataset Structure & Count Files

**Purpose:**
* Verify that the uploaded/extracted dataset has the correct YOLO structure and count all files/folders

**Details:**
* Calls `verify_dataset_structure()` function (defined in Section 3: Helper Functions)
* Checks for `dataset.yaml` file
* Verifies required directories exist: `images/train`, `images/val`, `labels/train`, `labels/val`
* **Counts all files and subfolders** in the complete dataset structure:
  * `yolo_dataset/`
  * `├── dataset.yaml`
  * `├── images/` (train, val, test)
  * `└── labels/` (train, val, test)
* Reports detailed file counts for each directory and subdirectory

**Note:** Run Section 3 (Helper Functions) first to define the function, then run this cell after uploading the dataset.


In [12]:
# Verify dataset structure and count all files/folders
# Note: verify_dataset_structure() function is defined in Section 3: Helper Functions

import os
import shutil

DATASET_EXTRACT_PATH = "/content/yolo_dataset"

if os.path.exists(DATASET_EXTRACT_PATH):

    # ============================================================
    # CLEANUP: Fix nested structure and remove __MACOSX
    # ============================================================
    print("\n🔍 Checking for nested folders and macOS junk...")

    nested_dataset_path = os.path.join(DATASET_EXTRACT_PATH, "yolo_dataset")
    macosx_path = os.path.join(DATASET_EXTRACT_PATH, "__MACOSX")

    # Fix nested yolo_dataset folder
    if os.path.exists(nested_dataset_path) and os.path.isdir(nested_dataset_path):
        print("⚠️  Found nested 'yolo_dataset' folder")
        print("📦 Moving files up one level...")

        # Move each item from nested folder to parent
        for item in os.listdir(nested_dataset_path):
            src = os.path.join(nested_dataset_path, item)
            dst = os.path.join(DATASET_EXTRACT_PATH, item + "_temp")
            shutil.move(src, dst)

        # Remove now-empty nested folder
        os.rmdir(nested_dataset_path)

        # Rename temp items to final names
        for item in os.listdir(DATASET_EXTRACT_PATH):
            if item.endswith("_temp"):
                src = os.path.join(DATASET_EXTRACT_PATH, item)
                dst = os.path.join(DATASET_EXTRACT_PATH, item.replace("_temp", ""))
                shutil.move(src, dst)

        print("✅ Files moved successfully!")

    # Remove __MACOSX folder
    if os.path.exists(macosx_path):
        print("🗑️  Removing __MACOSX folder...")
        shutil.rmtree(macosx_path)
        print("✅ __MACOSX removed!")

    # Remove ZIP file if still present
    zip_file = os.path.join(DATASET_EXTRACT_PATH, "yolo_dataset.zip")
    if os.path.exists(zip_file):
        print("🗑️  Removing yolo_dataset.zip...")
        os.remove(zip_file)
        print("✅ ZIP file removed!")

    print("✅ Cleanup complete!\n")

    # ============================================================
    # VERIFICATION
    # ============================================================

    # First verify structure
    dataset_valid = verify_dataset_structure(DATASET_EXTRACT_PATH)

    if dataset_valid:
        print(f"\n✅ Dataset ready at: {DATASET_EXTRACT_PATH}")

        # Count all files and folders
        print(f"\n📊 === DATASET FILE & FOLDER COUNT ===")
        print("="*60)

        def count_files_and_folders(root_path, relative_path=""):
            """Recursively count files and folders"""
            full_path = os.path.join(root_path, relative_path) if relative_path else root_path
            display_path = relative_path if relative_path else "yolo_dataset/"

            if not os.path.exists(full_path):
                return 0, 0

            file_count = 0
            folder_count = 0

            try:
                items = os.listdir(full_path)
                for item in items:
                    item_path = os.path.join(full_path, item)
                    item_display = os.path.join(display_path, item) if display_path else item

                    if os.path.isfile(item_path):
                        file_count += 1
                    elif os.path.isdir(item_path):
                        folder_count += 1
                        # Recursively count subdirectories
                        sub_files, sub_folders = count_files_and_folders(root_path, os.path.join(relative_path, item) if relative_path else item)
                        file_count += sub_files
                        folder_count += sub_folders
            except PermissionError:
                pass

            return file_count, folder_count

        # Count for each main directory
        total_files = 0
        total_folders = 0

        # Root level
        root_files = len([f for f in os.listdir(DATASET_EXTRACT_PATH) if os.path.isfile(os.path.join(DATASET_EXTRACT_PATH, f))])
        root_folders = len([f for f in os.listdir(DATASET_EXTRACT_PATH) if os.path.isdir(os.path.join(DATASET_EXTRACT_PATH, f))])
        print(f"📁 yolo_dataset/")
        print(f"   Files: {root_files}")
        print(f"   Folders: {root_folders}")
        total_files += root_files
        total_folders += root_folders

        # Check for dataset.yaml
        dataset_yaml = os.path.join(DATASET_EXTRACT_PATH, "dataset.yaml")
        if os.path.exists(dataset_yaml):
            print(f"   ✅ dataset.yaml")

        # Count images directory
        images_path = os.path.join(DATASET_EXTRACT_PATH, "images")
        if os.path.exists(images_path):
            print(f"\n📁 images/")
            for subdir in ["train", "val", "test"]:
                subdir_path = os.path.join(images_path, subdir)
                if os.path.exists(subdir_path):
                    subdir_files = len([f for f in os.listdir(subdir_path) if os.path.isfile(os.path.join(subdir_path, f))])
                    print(f"   📁 {subdir}/: {subdir_files} files")
                    total_files += subdir_files
                else:
                    print(f"   📁 {subdir}/: (not found)")

        # Count labels directory
        labels_path = os.path.join(DATASET_EXTRACT_PATH, "labels")
        if os.path.exists(labels_path):
            print(f"\n📁 labels/")
            for subdir in ["train", "val", "test"]:
                subdir_path = os.path.join(labels_path, subdir)
                if os.path.exists(subdir_path):
                    subdir_files = len([f for f in os.listdir(subdir_path) if os.path.isfile(os.path.join(subdir_path, f))])
                    print(f"   📁 {subdir}/: {subdir_files} files")
                    total_files += subdir_files
                else:
                    print(f"   📁 {subdir}/: (not found)")

        # Total count
        print("="*60)
        print(f"📊 TOTAL SUMMARY:")
        print(f"   Total Files: {total_files}")
        print(f"   Total Folders: {total_folders + root_folders}")
        print("="*60)

    else:
        print(f"\n⚠️  Dataset structure may be incomplete. Please check the paths.")
else:
    print(f"\n❌ Dataset not found at: {DATASET_EXTRACT_PATH}")
    print(f"   Please run Section 4.1 (Upload Dataset) first.")


🔍 Checking for nested folders and macOS junk...
⚠️  Found nested 'yolo_dataset' folder
📦 Moving files up one level...
✅ Files moved successfully!
🗑️  Removing __MACOSX folder...
✅ __MACOSX removed!
✅ Cleanup complete!


🔍 Verifying dataset structure at: /content/yolo_dataset
✅ Found dataset.yaml
   Dataset YAML appears valid
✅ images/train: 23754 files
✅ images/val: 2332 files
✅ labels/train: 23754 files
✅ labels/val: 2332 files
✅ Dataset structure is valid!

✅ Dataset ready at: /content/yolo_dataset

📊 === DATASET FILE & FOLDER COUNT ===
📁 yolo_dataset/
   Files: 2
   Folders: 2
   ✅ dataset.yaml

📁 images/
   📁 train/: 23754 files
   📁 val/: 2332 files
   📁 test/: 3914 files

📁 labels/
   📁 train/: 23754 files
   📁 val/: 2332 files
   📁 test/: 3914 files
📊 TOTAL SUMMARY:
   Total Files: 60002
   Total Folders: 4


## Section 6: Upload Model File

**Purpose:**
* Upload the pretrained YOLO12n model file for training

**Details:**
* Prompts user to upload `yolo12n.pt` model file
* Saves uploaded model to `/content/models/` directory
* If model is not uploaded, YOLO will auto-download it during training
* Recommended to upload if you have a specific pretrained model version


In [14]:
# Upload pretrained model file
from google.colab import files
import os
import shutil

UPLOAD_MODEL = True  # Set to False if model already exists or want to auto-download
MODELS_DIR = "/content/models"  # Directory to store model files
MODEL_FILENAME = "yolo12n.pt"  # Model filename

if UPLOAD_MODEL:
    os.makedirs(MODELS_DIR, exist_ok=True)

    print("📤 Please upload the pretrained model file:")
    print(f"   Model file: {MODEL_FILENAME}")
    print(f"   Expected path: /Users/jvarghese/Documents/TrafficSignProject/model/pre-trained-models/yolo12n.pt")
    print()
    print("   Note: If you skip this, YOLO will auto-download yolo12n.pt during training")
    print()

    uploaded_models = files.upload()

    if uploaded_models:
        print(f"✅ Uploaded {len(uploaded_models)} file(s)")

        # Move uploaded files to models directory
        # Always save as yolo12n.pt for consistency
        for filename in uploaded_models.keys():
            if filename.endswith('.pt'):
                # Save as yolo12n.pt regardless of uploaded filename
                dest_path = os.path.join(MODELS_DIR, MODEL_FILENAME)
                shutil.move(filename, dest_path)
                print(f"   ✅ Saved as: {dest_path}")
            else:
                print(f"   ⚠️  Skipping non-.pt file: {filename}")

        # Set model path for later use
        MODEL_UPLOAD_PATH = os.path.join(MODELS_DIR, MODEL_FILENAME)
        if os.path.exists(MODEL_UPLOAD_PATH):
            print(f"\n✅ Model ready at: {MODEL_UPLOAD_PATH}")
        else:
            print(f"\n⚠️  Model file not found at expected path")
            MODEL_UPLOAD_PATH = None
    else:
        print("⚠️  No model file uploaded.")
        print("   YOLO will auto-download yolo12n.pt during training if not found")
        MODEL_UPLOAD_PATH = None
else:
    print("⏭️  Skipping model upload (UPLOAD_MODEL=False)")
    MODEL_UPLOAD_PATH = os.path.join(MODELS_DIR, MODEL_FILENAME)
    if os.path.exists(MODEL_UPLOAD_PATH):
        print(f"✅ Using existing model at: {MODEL_UPLOAD_PATH}")
    else:
        print(f"⚠️  Model not found at: {MODEL_UPLOAD_PATH}")
        print("   YOLO will auto-download yolo12n.pt during training")
        MODEL_UPLOAD_PATH = None

📤 Please upload the pretrained model file:
   Model file: yolo12n.pt
   Expected path: /Users/jvarghese/Documents/TrafficSignProject/model/pre-trained-models/yolo12n.pt

   Note: If you skip this, YOLO will auto-download yolo12n.pt during training



Saving yolo12n.pt to yolo12n.pt
✅ Uploaded 1 file(s)
   ✅ Saved as: /content/models/yolo12n.pt

✅ Model ready at: /content/models/yolo12n.pt


## Section 7: Configuration

**Purpose:**
* Set all training parameters, paths, and model configurations

**Details:**
* **Paths:** Dataset path, model path, project output directory
* **Training Parameters:** Epochs (200), batch size (32 for A100), learning rate (0.001), workers (16), image size (640)
* **Device:** CUDA (A100 GPU) - optimized for high-performance training
* **Mixed Precision:** Enabled (amp=True) for faster training on A100
* **Resume:** Automatically detects and uses last checkpoint if available


In [19]:
# === CONFIGURATION ===
# Update these paths for your environment
DATASET_BASE = "/content/yolo_dataset"  # Dataset path (set by Section 3)
MODEL_PATH = "/content/models/yolo12n.pt"  # Update for Colab
PROJECT_DIR = "/content/runs/train/traffic_signs_yolo12n_v1"
RESUME_CHECKPOINT = os.path.join(PROJECT_DIR, "weights", "last.pt")

# === OPTIMIZED CONFIGURATION FOR STANDARD RAM + A100 40GB ===
EPOCHS = 200
BATCH_SIZE = 64  # Increased from 32
LEARNING_RATE = 0.001
WORKERS = 24  # Increased from 16 (50% more parallel loading)
IMAGE_SIZE = 640  # Keep standard size

# Device selection - A100 GPU (CUDA)
if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"✅ Using CUDA device: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = "cpu"
    print("⚠️  Warning: CUDA not available, falling back to CPU (not recommended)")

# Mixed Precision Training - Enabled for A100
USE_AMP = True  # Automatic Mixed Precision for faster training on A100

print(f"\n📁 Dataset: {DATASET_BASE}")
print(f"📁 Model: {MODEL_PATH}")
print(f"📁 Output Directory: {PROJECT_DIR}")
print(f"🔧 Device: {DEVICE}")
print(f"⚡ Mixed Precision (AMP): {USE_AMP}")
print(f"📊 Epochs: {EPOCHS}")
print(f"📦 Batch Size: {BATCH_SIZE}")
print(f"👷 Workers: {WORKERS}")
print(f"📈 Learning Rate: {LEARNING_RATE}")
print(f"🖼️  Image Size: {IMAGE_SIZE}")


✅ Using CUDA device: NVIDIA A100-SXM4-40GB

📁 Dataset: /content/yolo_dataset
📁 Model: /content/models/yolo12n.pt
📁 Output Directory: /content/runs/train/traffic_signs_yolo12n_v1
🔧 Device: cuda
⚡ Mixed Precision (AMP): True
📊 Epochs: 200
📦 Batch Size: 64
👷 Workers: 24
📈 Learning Rate: 0.001
🖼️  Image Size: 640


## Section 8: Divergence Monitor Class

**Purpose:**
* Monitor training metrics and report divergence information (information only, no actions)

**Details:**
* `DivergenceMonitor` class tracks training progress and reports every 5 epochs
* **Monitoring:** Tracks mAP@0.5 values, training loss trends, and performance divergence
* **Reporting:** Provides status indicators (well-performing, minor divergence, notable divergence)
* **Information Only:** Does not take any actions - YOLO's built-in patience handles early stopping
* Reports divergence between current mAP and recent best mAP to help identify overfitting trends


In [20]:
class DivergenceMonitor:
    """Simple divergence monitor - information only, no actions taken"""

    def __init__(self, report_interval=5):
        self.report_interval = report_interval
        self.last_report = 0

    def on_train_epoch_end(self, trainer):
        """Monitor divergence every 5 epochs - INFORMATION ONLY"""
        epoch = trainer.epoch + 1

        # Report every 5 epochs
        if epoch >= 5 and (epoch - self.last_report >= self.report_interval):
            self.last_report = epoch

            try:
                # Get results CSV path
                results_path = os.path.join(trainer.save_dir, "results.csv")

                if os.path.exists(results_path):
                    # Read training results
                    df = pd.read_csv(results_path)

                    if len(df) >= 10:  # Need at least 10 epochs for meaningful analysis
                        # Get recent validation mAP50 values
                        recent_map50 = df['val/mAP50(B)'].iloc[-10:].values if 'val/mAP50(B)' in df.columns else df['metrics/mAP50(B)'].iloc[-10:].values
                        current_map50 = recent_map50[-1]
                        best_in_recent = np.max(recent_map50)
                        divergence = best_in_recent - current_map50

                        # Get training loss trend
                        recent_loss = df['train/box_loss'].iloc[-5:].values if 'train/box_loss' in df.columns else [0]
                        loss_trend = "↗️ Rising" if len(recent_loss) > 1 and recent_loss[-1] > recent_loss[0] else "↘️ Falling"

                        print(f"\n📊 === DIVERGENCE MONITOR - Epoch {epoch} ===")
                        print(f"   🎯 Current mAP@0.5: {current_map50:.4f}")
                        print(f"   🏆 Recent best mAP@0.5: {best_in_recent:.4f}")
                        print(f"   📈 Performance divergence: {divergence:.4f}")
                        print(f"   📉 Training loss trend: {loss_trend}")

                        # Simple status indicators
                        if divergence < 0.01:
                            print(f"   ✅ Status: Model performing well (low divergence)")
                        elif divergence < 0.05:
                            print(f"   ⚠️  Status: Minor divergence detected (monitoring)")
                        else:
                            print(f"   🔍 Status: Notable divergence (information only)")

                        print(f"   ℹ️  Note: This is informational only - no actions taken")
                        print(f"   🛡️  YOLO patience will handle early stopping at {trainer.args.patience} epochs")

                    else:
                        # Not enough data yet
                        current_map50 = df['val/mAP50(B)'].iloc[-1] if 'val/mAP50(B)' in df.columns and len(df) > 0 else 0
                        print(f"\n📊 === DIVERGENCE MONITOR - Epoch {epoch} ===")
                        print(f"   🎯 Current mAP@0.5: {current_map50:.4f}")
                        print(f"   ⏳ Building history... need 10+ epochs for divergence analysis")

            except Exception as e:
                print(f"\n📊 === DIVERGENCE MONITOR - Epoch {epoch} ===")
                print(f"   ⚠️ Unable to read training data: {e}")
                print(f"   ℹ️ Monitor will retry next reporting interval")

    def on_train_start(self, trainer):
        """Initialize monitoring"""
        print(f"\n🔍 DIVERGENCE MONITOR INITIALIZED")
        print(f"   📊 Reporting interval: Every {self.report_interval} epochs")
        print(f"   ℹ️ Information only - no training interventions")
        print(f"   🛡️ YOLO patience: {trainer.args.patience} epochs")
        print()

    def on_train_end(self, trainer):
        """Training completed"""
        print(f"\n✅ DIVERGENCE MONITORING COMPLETE")
        print(f"   📊 All decisions handled by YOLO's built-in mechanisms")


## Section 9: Training Function

**Purpose:**
* Execute the complete YOLO training pipeline with divergence monitoring

**Details:**
* `train_traffic_signs()` function orchestrates the entire training process
* **Setup:** Validates dataset and model paths, clears caches, creates output directories
* **Resume Logic:** Automatically detects and resumes from last checkpoint if available
* **Model Loading:** Loads YOLO model (from checkpoint or pre-trained model)
* **Monitoring:** Registers divergence monitor callbacks
* **Training:** Runs training with validation enabled, all visualizations enabled for academic work
* **Mixed Precision:** Enabled (amp=True) for faster training on A100 GPU
* **Augmentation:** All augmentation disabled for maximum speed and simplicity
* Returns True on success, False on failure


In [21]:
def train_traffic_signs():
    """Ultra-simplified YOLO training"""

    # Dataset YAML path
    dataset_yaml = os.path.join(DATASET_BASE, "dataset.yaml")

    if not os.path.exists(dataset_yaml):
        print(f"❌ Dataset YAML not found: {dataset_yaml}")
        return False

    if not os.path.exists(MODEL_PATH):
        print(f"❌ Model not found: {MODEL_PATH}")
        return False

    # Clear caches and GPU memory
    print("\n🧹 Clearing caches and GPU memory...")
    delete_cache_files()
    clear_gpu_cache()
    print()

    # Create project directory
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.makedirs(os.path.join(PROJECT_DIR, "weights"), exist_ok=True)

    # Load model (resume if checkpoint exists)
    resume_training = os.path.exists(RESUME_CHECKPOINT)
    if resume_training:
        print(f"🔄 Resuming from: {RESUME_CHECKPOINT}")
        model = YOLO(RESUME_CHECKPOINT)
    else:
        print(f"🆕 Starting new training with: {MODEL_PATH}")
        model = YOLO(MODEL_PATH)

    print(f"🚀 Starting training...")
    print(f"   - Epochs: {EPOCHS}")
    print(f"   - Batch size: {BATCH_SIZE}")
    print(f"   - Learning rate: {LEARNING_RATE}")
    print(f"   - Workers: {WORKERS}")
    print(f"   - Device: {DEVICE}")
    print(f"   - Image size: {IMAGE_SIZE}")
    print()

    # Initialize divergence monitor
    monitor = DivergenceMonitor(report_interval=5)

    # Register callbacks
    model.add_callback("on_train_start", monitor.on_train_start)
    model.add_callback("on_train_epoch_end", monitor.on_train_epoch_end)
    model.add_callback("on_train_end", monitor.on_train_end)

    # Train with minimal settings
    results = model.train(
        # Data
        data=dataset_yaml,

        # Resume training if checkpoint exists
        resume=resume_training,

        # Training parameters
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMAGE_SIZE,
        device=DEVICE,
        workers=WORKERS,

        # Learning settings
        lr0=LEARNING_RATE,
        optimizer='AdamW',
        cos_lr=True,     # Smooth learning rate decay
        lrf=0.1,         # Conservative final LR (0.0001)

        # Project settings
        project=os.path.dirname(PROJECT_DIR),
        name=os.path.basename(PROJECT_DIR),
        exist_ok=True,

        # Save settings
        save_period=1,  # Save every epoch

        # Validation settings - ENABLED
        val=True,  # Enable validation during training

        # Plotting settings - ENABLED (all outputs)
        plots=True,  # Enable plots
        save_json=True,  # Save validation results as JSON
        save_conf=True,  # Save confidences in labels
        save_crop=True,  # Save cropped prediction boxes

        # Data augmentation - DISABLED
        hsv_h=0.0,
        hsv_s=0.0,
        hsv_v=0.0,
        degrees=0.0,
        translate=0.0,
        scale=0.0,
        shear=0.0,
        perspective=0.0,
        flipud=0.0,
        fliplr=0.0,
        mosaic=0.0,
        mixup=0.0,
        copy_paste=0.0,

        # Stability settings
        cache=True,  # Enable dataset caching for A100 (large GPU memory)
        rect=False,
        single_cls=False,
        amp=True,  # Mixed precision for A100 GPU

        # Early stopping - ENABLED (with validation)
        patience=20,  # Stop if no improvement for 20 epochs

        # Other settings
        verbose=True,
        seed=42,
        deterministic=True
    )

    print("\n✅ Training completed!")
    print(f"📁 Results saved to: {PROJECT_DIR}")

    return True


## Section 10: Run Training

**Purpose:**
* Execute the complete training pipeline

**Details:**
* Checks CUDA availability (A100 GPU)
* Calls `train_traffic_signs()` to start training
* Training will automatically:
  * Resume from last checkpoint if available
  * Save checkpoints every epoch
  * Monitor divergence every 5 epochs (information only)
  * Stop early if no improvement for 20 epochs (YOLO patience)
* **Validation Enabled:** Validation runs during training to monitor progress
* **Visualizations Enabled for Academic Work:**
  * Training/validation plots (loss curves, mAP curves, etc.)
  * Validation results in JSON format
  * Confidence scores saved
  * Cropped detections saved for visualization
* Prints success/failure message at completion
* All results saved to project directory


In [23]:
# Start training
print("🎯 ULTRA-SIMPLIFIED YOLO TRAINING - A100 GPU OPTIMIZED")
print("=" * 60)

# Check GPU availability
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ CUDA (GPU) available: {gpu_name}")
    print(f"✅ GPU Memory: {gpu_memory:.2f} GB")
    print(f"✅ Mixed Precision (AMP): Enabled")
    print(f"✅ Dataset Caching: Enabled")
else:
    print("❌ CUDA not available - A100 GPU required for optimal performance")
    print("⚠️  Falling back to CPU (not recommended)")

print()

# Run training
success = train_traffic_signs()

if success:
    print("\n🎉 Training completed successfully!")
    print(f"📁 Check results in: {PROJECT_DIR}")
    print(f"🏆 Best model: {PROJECT_DIR}/weights/best.pt")
    print(f"🔄 Resume from: {PROJECT_DIR}/weights/last.pt")
    print(f"\n📊 Academic Visualizations Available:")
    print(f"   - Training plots: {PROJECT_DIR}/results.png")
    print(f"   - Validation JSON: {PROJECT_DIR}/predictions/")
    print(f"   - Confidence scores: {PROJECT_DIR}/labels/")
    print(f"   - Cropped detections: {PROJECT_DIR}/predictions/")
else:
    print("\n💥 Training failed!")


🎯 ULTRA-SIMPLIFIED YOLO TRAINING - A100 GPU OPTIMIZED
✅ CUDA (GPU) available: NVIDIA A100-SXM4-40GB
✅ GPU Memory: 42.47 GB
✅ Mixed Precision (AMP): Enabled
✅ Dataset Caching: Enabled


🧹 Clearing caches and GPU memory...
✅ CUDA cache cleared
✅ System memory cleaned

🆕 Starting new training with: /content/models/yolo12n.pt
🚀 Starting training...
   - Epochs: 200
   - Batch size: 64
   - Learning rate: 0.001
   - Workers: 24
   - Device: cuda
   - Image size: 640

Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, ex

## Section 11: Validation with Best Model

**Purpose:**
* Validate the trained model using the best checkpoint with comprehensive visualizations

**Details:**
* Loads the best.pt model from training output
* Runs validation on the validation dataset
* **Saves results to separate validation directory:** `/content/runs/val/traffic_signs_yolo12n_v1/`
* Displays comprehensive metrics: mAP@0.5, mAP@0.5:0.95, precision, recall
* Shows per-class performance metrics (top 10 and bottom 10 classes)

**Generated Outputs:**
* **Standard YOLO Plots:** Confusion matrix (normalized & unnormalized), PR curve, F1 curve, P curve, R curve
* **Classification Analysis:** Per-class performance bar chart, performance distribution histogram, top/bottom classes comparison
* **Prediction Visualizations:** Sample images with predicted bounding boxes and labels, validation batch images with labels
* **Summary Grid:** 6 sample predictions in a single visualization
* **Data Outputs:** JSON results (COCO format), CSV metrics, per-class metrics CSV, prediction labels with confidence scores
* **Individual Predictions:** Detailed prediction images for each sample
* **Performance Statistics:** Mean, median, std dev, min, max mAP@0.5 across all classes

**Note:** All validation results are saved separately from training results for clarity


In [24]:
# Validation with best.pt model
print("🔍 VALIDATION: Testing Best Model Performance")
print("=" * 60)

# Path to best model
BEST_MODEL_PATH = os.path.join(PROJECT_DIR, "weights", "best.pt")

# Create validation output directory (separate from training)
VALIDATION_DIR = "/content/runs/val/traffic_signs_yolo12n_v1"
os.makedirs(VALIDATION_DIR, exist_ok=True)

# Check if best.pt exists
if not os.path.exists(BEST_MODEL_PATH):
    print(f"❌ Best model not found at: {BEST_MODEL_PATH}")
    print("   Please run training first (Section 10)")
else:
    print(f"✅ Loading best model: {BEST_MODEL_PATH}")
    print(f"✅ Validation results will be saved to: {VALIDATION_DIR}\n")

    # Load the best model
    best_model = YOLO(BEST_MODEL_PATH)

    # Dataset YAML path
    dataset_yaml = os.path.join(DATASET_BASE, "dataset.yaml")

    print("🚀 Running validation on validation dataset...")
    print(f"   Dataset: {dataset_yaml}")
    print(f"   Device: {DEVICE}")
    print(f"   Image Size: {IMAGE_SIZE}")
    print(f"   Output: {VALIDATION_DIR}")
    print()

    # Run validation with separate output directory - COMPREHENSIVE MODE
    try:
        validation_results = best_model.val(
            data=dataset_yaml,
            imgsz=IMAGE_SIZE,
            batch=BATCH_SIZE,
            device=DEVICE,
            workers=WORKERS,
            conf=0.25,  # Confidence threshold
            iou=0.45,   # IoU threshold for NMS
            plots=True,  # Generate ALL validation plots
            save_json=True,  # Save results as JSON (COCO format)
            save_txt=True,  # Save prediction labels
            save_conf=True,  # Save confidence scores in labels
            save_crop=False,  # Don't save cropped predictions (too many files)
            project=os.path.dirname(VALIDATION_DIR),  # Save to val directory
            name=os.path.basename(VALIDATION_DIR),
            exist_ok=True,
            verbose=True,
            # Additional metrics
            rect=False,  # Evaluate at native resolution
            split='val'  # Use validation split
        )

        print("\n" + "=" * 60)
        print("📊 VALIDATION RESULTS")
        print("=" * 60)

        # Display key metrics
        print(f"\n🎯 Overall Performance:")
        print(f"   mAP@0.5:      {validation_results.results_dict.get('metrics/mAP50(B)', 0):.4f}")
        print(f"   mAP@0.5:0.95: {validation_results.results_dict.get('metrics/mAP50-95(B)', 0):.4f}")
        print(f"   Precision:    {validation_results.results_dict.get('metrics/precision(B)', 0):.4f}")
        print(f"   Recall:       {validation_results.results_dict.get('metrics/recall(B)', 0):.4f}")

        # Display per-class metrics if available
        if hasattr(validation_results, 'ap_class_index') and validation_results.ap_class_index is not None:
            print(f"\n📋 Per-Class Performance:")

            # Get class names from dataset.yaml
            import yaml
            with open(dataset_yaml, 'r') as f:
                dataset_config = yaml.safe_load(f)
                class_names = dataset_config.get('names', [])

            # Display top 10 and bottom 10 classes by mAP
            if hasattr(validation_results, 'ap50') and validation_results.ap50 is not None:
                ap50_values = validation_results.ap50

                # Sort classes by mAP@0.5
                class_performance = []
                for idx, ap in enumerate(ap50_values):
                    class_name = class_names[idx] if idx < len(class_names) else f"Class {idx}"
                    class_performance.append((class_name, ap))

                class_performance.sort(key=lambda x: x[1], reverse=True)

                print(f"\n   Top 10 Classes (by mAP@0.5):")
                for i, (class_name, ap) in enumerate(class_performance[:10]):
                    print(f"      {i+1}. {class_name}: {ap:.4f}")

                if len(class_performance) > 10:
                    print(f"\n   Bottom 10 Classes (by mAP@0.5):")
                    for i, (class_name, ap) in enumerate(class_performance[-10:]):
                        print(f"      {len(class_performance)-9+i}. {class_name}: {ap:.4f}")

        print("\n" + "=" * 60)
        print(f"✅ Validation completed successfully!")
        print(f"📁 Validation results saved to: {VALIDATION_DIR}/")
        print("=" * 60)

        # List all generated plots and outputs
        print(f"\n📊 Generated Validation Outputs:")
        print(f"\n   🎯 Performance Curves:")
        print(f"      📈 Confusion Matrix: confusion_matrix.png")
        print(f"      📈 Confusion Matrix (normalized): confusion_matrix_normalized.png")
        print(f"      📈 Precision-Recall Curve: PR_curve.png")
        print(f"      📈 F1 Score Curve: F1_curve.png")
        print(f"      📈 Precision Curve: P_curve.png")
        print(f"      📈 Recall Curve: R_curve.png")
        print(f"\n   📊 Classification Metrics:")
        print(f"      📋 Results CSV: results.csv")
        print(f"      📋 JSON Results (COCO format): predictions.json")
        print(f"      📄 Prediction Labels: labels/ (with confidence scores)")
        print(f"\n   🖼️  Visual Outputs:")
        print(f"      🖼️  Validation Batch Images: val_batch*_pred.jpg")
        print(f"      🖼️  Validation Batch Labels: val_batch*_labels.jpg")
        print(f"\n📈 Training results (for comparison): {PROJECT_DIR}/")

        # Generate additional classification analysis plots
        print(f"\n📊 Generating additional classification analysis...")

        # Get class names and metrics
        import yaml
        with open(dataset_yaml, 'r') as f:
            dataset_config = yaml.safe_load(f)
            class_names = dataset_config.get('names', [])

        # Create per-class performance analysis
        if hasattr(validation_results, 'box') and hasattr(validation_results.box, 'all_ap'):
            import matplotlib.pyplot as plt
            import seaborn as sns

            # Get per-class metrics
            ap50 = validation_results.box.all_ap[:, 0] if hasattr(validation_results.box, 'all_ap') else []
            ap75 = validation_results.box.all_ap[:, 5] if hasattr(validation_results.box, 'all_ap') and validation_results.box.all_ap.shape[1] > 5 else []

            if len(ap50) > 0:
                # Plot 1: Per-class mAP@0.5 bar chart
                fig, ax = plt.subplots(figsize=(16, 10))

                # Sort classes by mAP
                sorted_indices = ap50.argsort()[::-1]
                sorted_ap50 = ap50[sorted_indices]
                sorted_names = [class_names[i] if i < len(class_names) else f"Class {i}" for i in sorted_indices]

                colors = ['green' if ap > 0.7 else 'orange' if ap > 0.5 else 'red' for ap in sorted_ap50]

                bars = ax.barh(range(len(sorted_ap50)), sorted_ap50, color=colors, alpha=0.7)
                ax.set_yticks(range(len(sorted_ap50)))
                ax.set_yticklabels(sorted_names, fontsize=8)
                ax.set_xlabel('mAP@0.5', fontsize=12, fontweight='bold')
                ax.set_title('Per-Class Performance (mAP@0.5) - Sorted by Performance', fontsize=14, fontweight='bold')
                ax.grid(axis='x', alpha=0.3)
                ax.set_xlim([0, 1])

                # Add value labels on bars
                for i, (bar, val) in enumerate(zip(bars, sorted_ap50)):
                    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                           f'{val:.3f}', va='center', fontsize=7)

                plt.tight_layout()
                class_perf_path = os.path.join(VALIDATION_DIR, "per_class_performance.png")
                plt.savefig(class_perf_path, dpi=150, bbox_inches='tight')
                plt.close()
                print(f"   ✅ Per-class performance chart: per_class_performance.png")

                # Plot 2: Performance distribution histogram
                fig, ax = plt.subplots(figsize=(10, 6))
                ax.hist(ap50, bins=20, color='skyblue', edgecolor='black', alpha=0.7)
                ax.axvline(ap50.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {ap50.mean():.3f}')
                ax.axvline(np.median(ap50), color='green', linestyle='--', linewidth=2, label=f'Median: {np.median(ap50):.3f}')
                ax.set_xlabel('mAP@0.5', fontsize=12, fontweight='bold')
                ax.set_ylabel('Number of Classes', fontsize=12, fontweight='bold')
                ax.set_title('Distribution of Per-Class Performance (mAP@0.5)', fontsize=14, fontweight='bold')
                ax.legend()
                ax.grid(alpha=0.3)
                plt.tight_layout()
                dist_path = os.path.join(VALIDATION_DIR, "performance_distribution.png")
                plt.savefig(dist_path, dpi=150, bbox_inches='tight')
                plt.close()
                print(f"   ✅ Performance distribution: performance_distribution.png")

                # Plot 3: Top 20 and Bottom 20 classes comparison
                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

                # Top 20
                top_20_indices = sorted_indices[:20]
                top_20_ap = ap50[top_20_indices]
                top_20_names = [class_names[i] if i < len(class_names) else f"Class {i}" for i in top_20_indices]

                ax1.barh(range(len(top_20_ap)), top_20_ap, color='green', alpha=0.7)
                ax1.set_yticks(range(len(top_20_ap)))
                ax1.set_yticklabels(top_20_names, fontsize=10)
                ax1.set_xlabel('mAP@0.5', fontsize=12, fontweight='bold')
                ax1.set_title('Top 20 Best Performing Classes', fontsize=14, fontweight='bold', color='green')
                ax1.grid(axis='x', alpha=0.3)
                ax1.set_xlim([0, 1])

                # Bottom 20
                bottom_20_indices = sorted_indices[-20:][::-1]
                bottom_20_ap = ap50[bottom_20_indices]
                bottom_20_names = [class_names[i] if i < len(class_names) else f"Class {i}" for i in bottom_20_indices]

                ax2.barh(range(len(bottom_20_ap)), bottom_20_ap, color='red', alpha=0.7)
                ax2.set_yticks(range(len(bottom_20_ap)))
                ax2.set_yticklabels(bottom_20_names, fontsize=10)
                ax2.set_xlabel('mAP@0.5', fontsize=12, fontweight='bold')
                ax2.set_title('Bottom 20 Worst Performing Classes', fontsize=14, fontweight='bold', color='red')
                ax2.grid(axis='x', alpha=0.3)
                ax2.set_xlim([0, 1])

                plt.tight_layout()
                comparison_path = os.path.join(VALIDATION_DIR, "top_bottom_classes_comparison.png")
                plt.savefig(comparison_path, dpi=150, bbox_inches='tight')
                plt.close()
                print(f"   ✅ Top/bottom classes comparison: top_bottom_classes_comparison.png")

                # Save detailed per-class metrics to CSV
                metrics_df = pd.DataFrame({
                    'Class': [class_names[i] if i < len(class_names) else f"Class {i}" for i in range(len(ap50))],
                    'mAP@0.5': ap50,
                    'mAP@0.75': ap75 if len(ap75) > 0 else [0] * len(ap50)
                })
                metrics_df = metrics_df.sort_values('mAP@0.5', ascending=False)
                metrics_csv_path = os.path.join(VALIDATION_DIR, "per_class_metrics.csv")
                metrics_df.to_csv(metrics_csv_path, index=False)
                print(f"   ✅ Per-class metrics CSV: per_class_metrics.csv")

                print(f"\n   📊 Performance Statistics:")
                print(f"      Mean mAP@0.5: {ap50.mean():.4f}")
                print(f"      Median mAP@0.5: {np.median(ap50):.4f}")
                print(f"      Std Dev: {ap50.std():.4f}")
                print(f"      Min: {ap50.min():.4f}")
                print(f"      Max: {ap50.max():.4f}")
                print(f"      Classes > 0.7: {(ap50 > 0.7).sum()} / {len(ap50)}")
                print(f"      Classes > 0.5: {(ap50 > 0.5).sum()} / {len(ap50)}")
                print(f"      Classes < 0.3: {(ap50 < 0.3).sum()} / {len(ap50)}")
        else:
            print(f"   ⚠️  Per-class metrics not available in validation results")

        # Generate sample predictions on validation images
        print(f"\n🖼️  Generating sample prediction visualizations...")

        # Get validation images
        val_images_dir = os.path.join(DATASET_BASE, "images", "val")
        if os.path.exists(val_images_dir):
            import random
            import cv2
            from PIL import Image
            import matplotlib.pyplot as plt

            # Get list of validation images
            val_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]

            if len(val_images) > 0:
                # Select 6 random images for visualization
                num_samples = min(6, len(val_images))
                sample_images = random.sample(val_images, num_samples)

                print(f"   Visualizing predictions on {num_samples} sample images...")

                # Create prediction output directory
                pred_vis_dir = os.path.join(VALIDATION_DIR, "sample_predictions")
                os.makedirs(pred_vis_dir, exist_ok=True)

                # Run predictions and save visualizations
                for idx, img_name in enumerate(sample_images):
                    img_path = os.path.join(val_images_dir, img_name)

                    # Run prediction
                    results = best_model.predict(
                        source=img_path,
                        conf=0.25,
                        iou=0.45,
                        save=True,
                        save_conf=True,
                        project=pred_vis_dir,
                        name=f"sample_{idx+1}",
                        exist_ok=True,
                        show_labels=True,
                        show_conf=True,
                        line_width=2
                    )

                print(f"   ✅ Sample predictions saved to: {pred_vis_dir}/")
                print(f"   📁 View individual predictions: {pred_vis_dir}/sample_*/")

                # Create a summary visualization
                print(f"\n📊 Creating prediction summary grid...")
                fig, axes = plt.subplots(2, 3, figsize=(15, 10))
                fig.suptitle('Sample Validation Predictions', fontsize=16, fontweight='bold')

                for idx in range(num_samples):
                    row = idx // 3
                    col = idx % 3
                    ax = axes[row, col]

                    # Find the predicted image
                    pred_img_path = os.path.join(pred_vis_dir, f"sample_{idx+1}", sample_images[idx])

                    if os.path.exists(pred_img_path):
                        img = Image.open(pred_img_path)
                        ax.imshow(img)
                        ax.set_title(f"Sample {idx+1}: {sample_images[idx][:20]}...", fontsize=10)
                        ax.axis('off')
                    else:
                        ax.text(0.5, 0.5, 'Image not found', ha='center', va='center')
                        ax.axis('off')

                plt.tight_layout()
                summary_path = os.path.join(VALIDATION_DIR, "prediction_summary.png")
                plt.savefig(summary_path, dpi=150, bbox_inches='tight')
                plt.close()

                print(f"   ✅ Prediction summary saved to: {summary_path}")
            else:
                print(f"   ⚠️  No validation images found in {val_images_dir}")
        else:
            print(f"   ⚠️  Validation images directory not found: {val_images_dir}")

        print("\n" + "=" * 60)
        print(f"📊 All validation outputs complete!")
        print(f"📁 Main directory: {VALIDATION_DIR}/")
        print("=" * 60)

    except Exception as e:
        print(f"\n❌ Validation failed: {e}")
        import traceback
        traceback.print_exc()
        print(f"   Please check the dataset and model paths")


🔍 VALIDATION: Testing Best Model Performance
✅ Loading best model: /content/runs/train/traffic_signs_yolo12n_v1/weights/best.pt
✅ Validation results will be saved to: /content/runs/val/traffic_signs_yolo12n_v1

🚀 Running validation on validation dataset...
   Dataset: /content/yolo_dataset/dataset.yaml
   Device: cuda
   Image Size: 640
   Output: /content/runs/val/traffic_signs_yolo12n_v1

Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
YOLOv12n summary (fused): 159 layers, 2,568,428 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1406.7±449.0 MB/s, size: 78.0 KB)
val: Scanning /content/yolo_dataset/labels/val.cache... 2332 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2332/2332 4.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 2.5it/s 14.5s
                   all       2332       2332      0.687      0.681      0.713

## Section 12: Backup Results to Google Drive

Copies entire `/content/runs` folder to Google Drive.

**Result:** `Google Drive → TrafficSignProject → model_outputs → runs/`

In [29]:
# Backup to Google Drive - SIMPLE VERSION
import shutil
import os
from google.colab import drive

drive.mount('/content/drive')

# Copy val
try:
    print("📤 Copying val...")
    shutil.copytree("/content/runs/val", "/content/drive/MyDrive/yolo_output/runs/val", dirs_exist_ok=True)
    print("✅ Val copied!")
except Exception as e:
    print(f"❌ Val error: {e}")

# Copy train
try:
    print("📤 Copying train...")
    shutil.copytree("/content/runs/train", "/content/drive/MyDrive/yolo_output/runs/train", dirs_exist_ok=True)
    print("✅ Train copied!")
except Exception as e:
    print(f"❌ Train error: {e}")

print("\n✅ Done! Check: Google Drive → yolo_output → runs/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📤 Copying val folder...
✅ Val folder copied!
📤 Copying train folder...
✅ Train folder copied!

✅ Done! Check: Google Drive → yolo_output → runs/
